# Cluster de semelhança das Justificativas

In [ ]:
# Instalação das bibliotecas vitais
# --- PROTOCOLO DE INSTALAÇÃO CORPORATIVA (TRAVA DE PROXY/SSL) ---
# 1. Instalação do UMAP
##!pip install umap-learn --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org
# 2. Instalação do HDBSCAN
##!pip install hdbscan --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org

In [1]:
# --- BIBLIOTECAS DE SUPORTE E MANIPULAÇÃO ---
import numpy as np                 # Operações matemáticas e manipulação de matrizes de embeddings
import pandas as pd                # Estruturação, limpeza e análise de dados tabulares (DataFrames)
# --- BIBLIOTECAS DE MODELAGEM/AGRUPAMENTO ---
import umap                        # Redução de dimensionalidade preservando a topologia e a semântica local
import hdbscan                     # Agrupamento (clustering) hierárquico baseado em densidade estatística

# --- CONFIGURAÇÕES DO AMBIENTE ---
import warnings
warnings.filterwarnings("ignore")                 # Suprime avisos do sistema (warnings) para garantir um output limpo
pd.set_option('display.max_columns', None)        # Garante a exibição visual de todas as colunas do DataFrame
pd.set_option('display.max_colwidth', None)       # Impede o truncamento do texto das justificativas longas na tela
pd.set_option('display.max_rows', 50)             # Define um limite prático de linhas para inspeção amostral eficiente

# 1 - Importação dos dados 

* Base completa com o escore de proximidade.

In [2]:
#Base completa
df_full = pd.read_csv('justificativa_outros_similaridades.csv', engine='python', encoding='utf-8', sep=',', on_bad_lines='skip')
#Volumetria
df_full.shape[0] #214.263

214263

In [3]:
#Informações sobre o data frame
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214263 entries, 0 to 214262
Data columns (total 30 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   prot_nu                      214263 non-null  int64  
 1   unidprot_cd                  214263 non-null  int64  
 2   nf_tp_registro               214263 non-null  int64  
 3   nf_sq_registro               214263 non-null  int64  
 4   cnpj                         214263 non-null  int64  
 5   inscricao_estadual           214263 non-null  int64  
 6   cnae_principal_numero        214263 non-null  int64  
 7   cnae_principal_descricao     214263 non-null  object 
 8   segmento_economico           214263 non-null  object 
 9   regime_contrib.              214263 non-null  object 
 10  ano_mes                      214263 non-null  object 
 11  gafdemanda_dt_geracao        214263 non-null  object 
 12  gafdemanda_dt_inicio_exec    36153 non-null   object 
 13 

## Matriz de vetorização

In [4]:
# 2. Carrega a matriz correspondente (subset de embeddings das órfãs)
embeddings_orfaos = np.load('embeddings_sefaz_full.npy') 

## Data frame dos órfãos

* Justificativas que não deram um match associado as opções disponíveis.

In [5]:
# --- FILTRAGEM COERENTE E ALINHAMENTO GEOMÉTRICO ---

# 1. Definição do critério rigoroso estabelecido
threshold_conservador = 0.68
filtro_orfaos = df_full['confianca_ia'] < threshold_conservador

# 2. Aplicação da máscara booleana simultânea (Garante o alinhamento do índice com o .npy)
df_orfaos = df_full[filtro_orfaos].copy()
embeddings_orfaos = embeddings_orfaos[filtro_orfaos.values]

# 3. Métricas de validação executiva
print(f"📊 Volume do DataFrame de Órfãos: {df_orfaos.shape[0]:,} linhas.")
print(f"📏 Volume da Matriz de Vetores:   {embeddings_orfaos.shape[0]:,} linhas.")

# Trava de segurança para validação acadêmica do Pós-doc
assert df_orfaos.shape[0] == embeddings_orfaos.shape[0], "❌ ERRO: Desalinhamento detectado entre texto e vetor!"
print("🎯 Sucesso: Alinhamento posicional validado com 100% de precisão. Pronto para o UMAP!")

📊 Volume do DataFrame de Órfãos: 122,975 linhas.
📏 Volume da Matriz de Vetores:   122,975 linhas.
🎯 Sucesso: Alinhamento posicional validado com 100% de precisão. Pronto para o UMAP!


# 2 - Redução da dimensionalidade

In [6]:
# --- CODIGO OTIMIZADO CONTRA TRAVAMENTO ---

print("📉 3. Executando redução dimensional via UMAP (384D -> 10D)...")
reducer = umap.UMAP(
    n_neighbors=30,
    min_dist=0.0,
    n_components=10,
    metric='cosine',
    random_state=42,
    low_memory=True,  # 🧠 Força o algoritmo a economizar RAM e evita Deadlocks
    verbose=True      # 📺 Exibe o progresso em tempo real na tela
)

# Executa o fit_transform monitorado
embeddings_reduzidos = reducer.fit_transform(embeddings_orfaos)

📉 3. Executando redução dimensional via UMAP (384D -> 10D)...
UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.0, n_components=10, n_jobs=1, n_neighbors=30, random_state=42, verbose=True)
Fri May 22 16:38:16 2026 Construct fuzzy simplicial set
Fri May 22 16:38:16 2026 Finding Nearest Neighbors
Fri May 22 16:38:16 2026 Building RP forest with 23 trees
Fri May 22 16:38:21 2026 NN descent for 17 iterations
	 1  /  17
	 2  /  17
	 3  /  17
	 4  /  17
	Stopping threshold met -- exiting after 4 iterations
Fri May 22 16:38:43 2026 Finished Nearest Neighbor Search
Fri May 22 16:38:46 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Fri May 22 19:31:41 2026 Finished embedding


# 3 - Cluster

In [8]:
#1. Caracterísicas do cluster
cluster = hdbscan.HDBSCAN(
    min_cluster_size = 200,
    metric = 'euclidean',
    cluster_selection_method = 'eom'
)
#2. Ajuste do modelo e gera os rótulos
cluster_labels = cluster.fit_predict(embeddings_reduzidos)
#3. Métricas
n_clusters = len(set(cluster_labels)) - (1 if - 1 in cluster_labels else 0)
n_ruido = list (cluster_labels).count(-1)
pct_ruido =  (n_ruido/len(cluster_labels)) * 100

#Visualização
print(f"Novos motivos fiscais identificados: {n_clusters}")
print(f"Justificativas classificadas como ruído (-1): {pct_ruido} {pct_ruido:.2f}%")

Novos motivos fiscais identificados: 65
Justificativas classificadas como ruído (-1): 57.78084976621265 57.78%


In [15]:
df_exploracao = pd.read_csv('justificativas_sem_razao_associada.csv', sep = ";")
df_exploracao['cluster_id'] = cluster_labels
#Salvar em CSV
df_exploracao.to_csv("tabela_exploracao.csv", index = False, encoding = 'utf-8')
#Salvar em EXCEL
df_exploracao.to_excel('tabela_exploracao.xlsx', index=False)